# Retention Simulation

Explore the Animal Company retention model: curve shape, DAU projections, and parameter sensitivity.

In [1]:
from IPython.display import HTML
display(HTML('''
<style id="aco-toggle-style"></style>
<button id="aco-toggle-btn"
  onclick="
    var style = document.getElementById('aco-toggle-style');
    if (style.innerHTML === '') {
      style.innerHTML = '.jp-Cell-inputWrapper { display: none !important; }';
      this.innerHTML = 'Show Code';
    } else {
      style.innerHTML = '';
      this.innerHTML = 'Hide Code';
    }
  "
  style="padding: 6px 16px; font-size: 13px; cursor: pointer;
         border: 1px solid #ccc; border-radius: 4px; background: #f5f5f5;">
  Hide Code
</button>
'''))

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Change to project root so relative paths work
os.chdir(Path(__file__).parent.parent if '__file__' in dir() else Path(__name__).resolve().parent if Path(__name__).exists() else Path.cwd())
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

from aco_model.models import RetentionCurve
from aco_model.retention import load_installs, retention_vector, simulate
from aco_model.config import load_config
from aco_model.state import save_state, load_state, DEFAULT_STATE_PATH

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Detect headless execution (nbconvert tests)
_HEADLESS = os.environ.get('ACO_HEADLESS') == '1'

# Load config and installs once
cfg = load_config()
installs = load_installs(cfg.installs_path)
print(f"Working directory: {Path.cwd()}")
print(f"Loaded {len(installs)} days of install data")
print(f"Retention anchors: {cfg.retention.anchors}")

Working directory: /home/spencer/dev/aco_model
Loaded 180 days of install data
Retention anchors: [(0, 100.0), (1, 41.0), (7, 22.0), (30, 5.5), (90, 1.5), (180, 0.0)]


## 1. Retention Curve

In [3]:
# Extract defaults from config anchors
_anchor_defaults = {d: r for d, r in cfg.retention.anchors}

d1_slider = widgets.FloatSlider(value=_anchor_defaults.get(1, 40), min=5, max=80, step=1, description='D1 %:')
d7_slider = widgets.FloatSlider(value=_anchor_defaults.get(7, 20), min=1, max=50, step=1, description='D7 %:')
d30_slider = widgets.FloatSlider(value=_anchor_defaults.get(30, 5), min=0.5, max=30, step=0.5, description='D30 %:')
d90_slider = widgets.FloatSlider(value=_anchor_defaults.get(90, 1), min=0.1, max=10, step=0.1, description='D90 %:')

def _reset_to_defaults(_):
    d1_slider.value = _anchor_defaults.get(1, 40)
    d7_slider.value = _anchor_defaults.get(7, 20)
    d30_slider.value = _anchor_defaults.get(30, 5)
    d90_slider.value = _anchor_defaults.get(90, 1)

def _save_as_defaults(_):
    import yaml
    _anchor_defaults[1] = d1_slider.value
    _anchor_defaults[7] = d7_slider.value
    _anchor_defaults[30] = d30_slider.value
    _anchor_defaults[90] = d90_slider.value
    cfg.retention = RetentionCurve(anchors=_anchors())
    config_path = Path('config.yaml')
    with open(config_path) as f:
        data = yaml.safe_load(f) or {}
    data['retention'] = {'anchors': [[a[0], round(a[1], 6)] for a in cfg.retention.anchors]}
    with open(config_path, 'w') as f:
        yaml.dump(data, f, default_flow_style=None, sort_keys=False)
    save_status.value = f'Saved: D1={d1_slider.value}%, D7={d7_slider.value}%, D30={d30_slider.value}%, D90={d90_slider.value}%'

reset_btn = widgets.Button(description='Reset to Defaults', button_style='warning',
                           layout=widgets.Layout(width='160px'))
reset_btn.on_click(_reset_to_defaults)
save_btn = widgets.Button(description='Save as Defaults', button_style='success',
                          layout=widgets.Layout(width='160px'))
save_btn.on_click(_save_as_defaults)
save_status = widgets.Label(value='')

def _anchors():
    return [(0, 100.0), (1, d1_slider.value), (7, d7_slider.value),
            (30, d30_slider.value), (90, d90_slider.value), (180, 0.0)]

# Output widgets for each section
out_curve = widgets.Output()
out_dau = widgets.Output()
out_proj = widgets.Output()
out_sens = widgets.Output()
out_heatmap = widgets.Output()

def _update_all(*args):
    d1, d7, d30, d90 = d1_slider.value, d7_slider.value, d30_slider.value, d90_slider.value
    anchors = _anchors()
    curve = RetentionCurve(anchors=anchors)

    # Section 1: Retention Curve
    with out_curve:
        out_curve.clear_output(wait=True)
        rates = retention_vector(181, curve)
        days = np.arange(0, 181)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(days, rates * 100, linewidth=2, color='#2196F3')
        ax1.set_xlabel('Days Since Install'); ax1.set_ylabel('Retention %')
        ax1.set_title('Retention Curve (Linear)')
        for d, r in [(1, d1), (7, d7), (30, d30), (90, d90)]:
            ax1.axhline(y=r, color='gray', linestyle='--', alpha=0.3)
            ax1.annotate(f'D{d}: {r:.1f}%', xy=(d, r), fontsize=9,
                         xytext=(d+10, r+3), arrowprops=dict(arrowstyle='->', color='gray'))
        ax2.semilogy(days[1:], rates[1:] * 100, linewidth=2, color='#FF5722')
        ax2.set_xlabel('Days Since Install'); ax2.set_ylabel('Retention % (log)')
        ax2.set_title('Retention Curve (Log Scale)'); ax2.set_ylim(0.1, 100)
        plt.tight_layout(); plt.show()

    # Section 2: DAU Simulation
    sim = simulate(installs, curve, cfg.sim_days)
    result = sim.to_dataframe()
    with out_dau:
        out_dau.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.fill_between(result['day'], result['dau'], alpha=0.3, color='#4CAF50', label='DAU')
        ax.plot(result['day'], result['dau'], linewidth=2, color='#4CAF50')
        ax.bar(result['day'], result['new_installs'], alpha=0.4, color='#2196F3', label='New Installs')
        ax.set_xlabel('Day'); ax.set_ylabel('Users')
        ax.set_title('DAU vs New Installs')
        ax.legend(loc='upper left')
        plt.tight_layout(); plt.show()
        avg_installs = result['new_installs'][result['new_installs'] > 0].mean()
        avg_dau = result['dau'].mean()
        print(f"Peak DAU: {result['dau'].max():,} (Day {result.loc[result['dau'].idxmax(), 'day']})")
        print(f"Avg Daily Installs: {avg_installs:,.0f}")
        print(f"Avg DAU: {avg_dau:,.0f}")
        print(f"Total Installs: {result['new_installs'].sum():,}")

    # Section 3: 365-Day Projection
    with out_proj:
        out_proj.clear_output(wait=True)
        last_day = int(installs.index[-1])
        last_rate = int(installs.iloc[-1])
        proj_days = 365
        extended_values = list(installs.values)
        for d in range(last_day + 1, proj_days + 1):
            extended_values.append(last_rate)
        extended = pd.Series(extended_values, index=np.arange(1, proj_days + 1), name='installs')
        extended.index.name = 'day'
        sim_365 = simulate(extended, curve, sim_days=proj_days)
        result_365 = sim_365.to_dataframe()

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.axvspan(1, last_day, alpha=0.05, color='blue', label='Actual install data')
        ax.axvspan(last_day, proj_days, alpha=0.05, color='orange', label=f'Projected ({last_rate:,}/day)')
        ax.fill_between(result_365['day'], result_365['dau'], alpha=0.3, color='#4CAF50')
        ax.plot(result_365['day'], result_365['dau'], linewidth=2, color='#4CAF50')
        ax.axvline(x=last_day, color='red', linestyle='--', alpha=0.7, label=f'End of install data (day {last_day})')
        ax.set_xlabel('Day'); ax.set_ylabel('DAU')
        ax.set_title('DAU Projection — 365 Days')
        ax.set_xlim(1, proj_days); ax.legend()
        plt.tight_layout(); plt.show()
        print(f"Steady-state DAU (Day {proj_days}): {result_365.iloc[-1]['dau']:,}")

    # Section 4: Sensitivity
    with out_sens:
        out_sens.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, 5))
        d1_values = sorted(set([30, 35, 40, 45, 50, d1]))
        for d1_val in d1_values:
            curve_var = RetentionCurve(anchors=[
                (0, 100.0), (1, d1_val), (7, d1_val * 0.5), (30, d1_val * 0.125),
                (90, d1_val * 0.025), (180, 0.0)
            ])
            r = simulate(installs, curve_var, sim_days=90).to_dataframe()
            style = {'linewidth': 3, 'linestyle': '--'} if d1_val == d1 else {'linewidth': 2}
            ax.plot(r['day'], r['dau'], label=f'D1={d1_val}%', **style)
        ax.set_xlabel('Day'); ax.set_ylabel('DAU')
        ax.set_title('DAU Sensitivity to D1 Retention')
        ax.legend(); plt.tight_layout(); plt.show()

    # Section 5: Cohort Heatmap
    with out_heatmap:
        out_heatmap.clear_output(wait=True)
        n_show = 30
        sim_hm = simulate(installs.iloc[:n_show], curve, sim_days=n_show)
        cdf = sim_hm.cohort_dataframe()
        fig, ax = plt.subplots(figsize=(12, 8))
        im = ax.imshow(cdf.values, aspect='auto', cmap='YlOrRd')
        ax.set_xlabel('Simulation Day'); ax.set_ylabel('Cohort (Install Day)')
        ax.set_title('Retained Users by Cohort')
        plt.colorbar(im, ax=ax, label='Active Users')
        plt.tight_layout(); plt.show()

    # Save state
    save_state(sim, anchors)

if _HEADLESS:
    _update_all()
else:
    for s in [d1_slider, d7_slider, d30_slider, d90_slider]:
        s.observe(lambda change: _update_all(), names='value')
    display(widgets.VBox([
        widgets.HBox([d1_slider, d7_slider, reset_btn, save_btn]),
        widgets.HBox([d30_slider, d90_slider, save_status]),
        out_curve
    ]))
    _update_all()

## 2. DAU Simulation (90 Days)

In [4]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_dau)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1200x500 with 2 Axes>', '…

## 3. Extended Projection (365 Days)

What happens if installs continue at the day-90 rate?

In [5]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_proj)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1200x500 with 1 Axes>', '…

## 4. Retention Sensitivity

How does DAU change if D1 retention shifts?

In [6]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_sens)

Output()

## 5. Cohort Heatmap

Retained users by cohort over time.

In [7]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_heatmap)

Output()